# `requests` 모듈로 FastAPI 호출 실습

> **사전 조건**: FastAPI 서버가 실행 중이어야 합니다.
> ```bash
> uvicorn news_api:app --reload
> ```

```
클라이언트 (Jupyter)
        ↓  HTTP 요청 (requests)
FastAPI 서버 (uvicorn)
        ↓  JSON 응답
클라이언트
```

## 1-1. GET 요청 기초

GET 요청: 서버에서 데이터를 **조회**할 때 사용합니다.  
URL에 파라미터를 붙이거나, 단순히 서버 상태를 확인할 때 주로 사용합니다.

In [8]:
import requests
import pandas as pd

BASE_URL = "http://127.0.0.1:8000"

# ── GET 요청 기본 구조 ────────────────────────────────────────────
response = requests.get(f"{BASE_URL}/")

# 응답 상태 코드 확인 (200: 성공, 404: 없음, 500: 서버 오류 등)
print("상태 코드:", response.status_code)

# 응답 본문을 딕셔너리로 변환 (.json() 메서드)
data = response.json()
print("응답 데이터:", data)
print("타입:", type(data))

상태 코드: 200
응답 데이터: {'message': '뉴스 헤드라인 카테고리 분류 API', 'docs': '/docs', 'model_loaded': True}
타입: <class 'dict'>


In [3]:
BASE_URL = "http://127.0.0.1:8000"

# ── GET 요청 + params: URL 쿼리 파라미터 전달 ─────────────────────
# 예) /search?keyword=python&limit=5
# params 딕셔너리를 넘기면 requests가 URL에 자동으로 붙여줍니다.

params = {"keyword": "python", "limit": 5}
response = requests.get(f"{BASE_URL}/search", params=params)

# 실제 요청이 어떤 URL로 날아갔는지 확인
print("요청 URL:", response.url)
# → http://127.0.0.1:8000/search?keyword=python&limit=5

# 응답 헤더에서 Content-Type 확인
print("Content-Type:", response.headers.get("Content-Type"))

요청 URL: http://127.0.0.1:8000/search?keyword=python&limit=5
Content-Type: application/json


In [9]:
# API 호출 예시

URL = 'https://api.openbrewerydb.org/v1/breweries'

response = requests.get(URL)

df = pd.json_normalize(response.json())

df.head()

,id,name,brewery_type,address_1,address_2,address_3,city,state_province,postal_code,country,longitude,latitude,phone,website_url,state,street
0,ae7b3174-8be8-4d53-a3a5-9b8240970eea,'s,brewpub,1 Friesener Straße,None,None,Kronach,Bayern,96317,Germany,11.327765,50.241246,+49 9261 628000,http://www.antla.de,Bayern,1 Friesener Straße
1,aa7cbe9b-3a0f-4888-9884-6186b0042b55,’t Drankorgel,micro,31a Geelsebaan,None,None,Mol,Vlaanderen,2400,Belgium,5.056955,51.176258,+32 496 76 40 63,http://tdrankorgel.be/,Vlaanderen,31a Geelsebaan
2,1034754f-abf9-4b42-bd6d-37e110ba4b1a,'T Kroontje,micro,62 Hogebrug,None,None,Lebbeke,Vlaanderen,9280,Belgium,4.091393,51.007888,+32 495 43 33 25,http://www.tkroontje.be/,Vlaanderen,62 Hogebrug
3,5128df48-79fc-4f0f-8b52-d06be54d0cec,(405) Brewing Co,micro,1716 Topeka St,None,None,Norman,Oklahoma,73069-8224,United States,-97.468182,35.257389,4058160490,http://www.405brewing.com,Oklahoma,1716 Topeka St
4,9c5a66c8-cc13-416f-a5d9-0a769c87d318,(512) Brewing Co,micro,407 Radam Ln Ste F200,None,None,Austin,Texas,78745-1197,United States,NaN,NaN,5129211545,http://www.512brewing.com,Texas,407 Radam Ln Ste F200


## 1-2. FastAPI 서버 상태 확인 (GET /health)

In [7]:
import requests

BASE_URL = "http://127.0.0.1:8000"

# /health 엔드포인트: 서버와 모델이 정상 로드되었는지 확인
response = requests.get(f"{BASE_URL}/health", timeout=3)
health = response.json()

print("서버 상태:", health["status"])      # "ok" 또는 "error"
print("상세 내용:", health["detail"])

if health["status"] == "ok":
    print("\n✅ 서버 정상 — 예측 요청 가능")
else:
    print("\n❌ 서버 오류 — 모델 파일을 확인하세요")

서버 상태: ok
상세 내용: model loaded

✅ 서버 정상 — 예측 요청 가능


## 1-3. POST 요청 기초

POST 요청: 서버에 데이터를 **전송**할 때 사용합니다.  
데이터는 URL이 아닌 **요청 본문(body)** 에 담아 보냅니다.  
FastAPI의 예측 엔드포인트처럼 입력값이 필요한 경우 주로 POST를 사용합니다.

In [6]:
import requests

BASE_URL = "http://127.0.0.1:8000"

# ── POST 요청: json 파라미터로 딕셔너리를 넘기면 자동으로 JSON 직렬화 ──
payload = {"headline": "Biden says U.S. forces would defend Taiwan if China invaded"}

response = requests.post(
    url=f"{BASE_URL}/predict",
    json=payload,          # json= 으로 넘기면 Content-Type: application/json 자동 설정
    timeout=10,            # 10초 안에 응답 없으면 예외 발생
)

print("상태 코드:", response.status_code)
print("응답 JSON:", response.json())

# ── 딕셔너리 키로 값 꺼내기 ───────────────────────────────────────
result = response.json()
print("\n예측 카테고리:", result["predicted_category"])
print("입력 헤드라인:", result["headline"])

상태 코드: 200
응답 JSON: {'headline': 'Biden says U.S. forces would defend Taiwan if China invaded', 'predicted_category': 'POLITICS'}

예측 카테고리: POLITICS
입력 헤드라인: Biden says U.S. forces would defend Taiwan if China invaded


## 1-4. 뉴스 헤드라인 카테고리 예측 (POST /predict)

In [ ]:
import requests

BASE_URL = "http://127.0.0.1:8000"

# 테스트할 헤드라인 목록
headlines = [
    "Biden says U.S. forces would defend Taiwan if China invaded",
    "Apple unveils new iPhone with AI-powered features",
    "Scientists discover new exoplanet that may support life",
    "Manchester United beats Arsenal in thrilling final",
    "Stock markets rally as inflation fears ease",
]

print(f"{'헤드라인':<55} {'예측 카테고리'}")
print("-" * 80)

for headline in headlines:
    response = requests.post(
        url=f"{BASE_URL}/predict",
        json={"headline": headline},
        timeout=10,
    )
    result = response.json()
    print(f"{result['headline'][:53]:<55} {result['predicted_category']}")